In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import palettable

import pathlib as pl

from tqdm.notebook import tqdm

In [ ]:
adata = sc.read_h5ad("/add/path/here/full_cohort.h5ad")

In [ ]:
refined_annotations = pd.read_csv("/add/path/here/refined_annotations.csv",index_col=0)

refined_annotations.columns = ["refined_annotations"]

In [ ]:
patient_id_mapping = {"CCG1153_4496262": "P1", "CCG1153_6640539": "P2", 
                      "CCG1153_4411": "P3", "Aguirre_EGSFR0074": "P4", 
                      "Aguirre_EGSFR0148": "P5", "Aguirre_EGSFR1732": "P6", 
                      "Aguirre_EGSFR0128": "P7", "Aguirre_EGSFR1938": "P8", 
                      "Aguirre_EGSFR1982": "P9", "Aguirre_EGSFR2218": "P10"}

In [ ]:
colorlist = palettable.colorbrewer.qualitative.Dark2_8.mpl_colors
colorlistbis = palettable.colorbrewer.qualitative.Paired_3.mpl_colors
colormapping_pat = {'Aguirre_EGSFR1982': colorlist[0], 
                    "Aguirre_EGSFR2218": colorlist[1], 
                    "CCG1153_4411": colorlist[2], 
                    "Aguirre_EGSFR1938": colorlist[3], 
                    "Aguirre_EGSFR0074": colorlist[4], 
                    "Aguirre_EGSFR0128": colorlist[5], 
                    "Aguirre_EGSFR1732": colorlist[6], 
                    "Aguirre_EGSFR0148": colorlist[7], 
                    "CCG1153_4496262": colorlistbis[0], 
                    "CCG1153_6640539": colorlistbis[1], 
                    "NA": "whitesmoke"}
colormapping_pat_bis = {patient_id_mapping[pat]: colormapping_pat[pat] for pat in patient_id_mapping}
colormapping_pat_bis["NA"] = "whitesmoke"

In [ ]:
colorlist = palettable.colorbrewer.qualitative.Set1_7.mpl_colors
colormapping_mal = {"cNMF_1": colorlist[0], "cNMF_2": colorlist[1], "cNMF_3": colorlist[3], 
                    "cNMF_4": colorlist[4], "cNMF_5": colorlist[6]}
colormapping_mal["Mixed"] = "lightgrey"
colormapping_mal["Outlier"] = "grey"

In [ ]:
highlevel_refined = {"Hepatocyte": "Epithelial", 
                     "Carcinoma": "Carcinoma", 
                     "Fibroblast": "Fibroblast", 
                     "Quiescent endothelial cells": "Endothelial", 
                     "Smooth muscle": "Muscle", 
                     "Skeletal muscle": "Muscle",
                     "TAM2": "Myeloid", "TAM3": "Myeloid",
                     "TCD4": "Lymphoid", 
                     "Inflammatory CAF": "Fibroblast", 
                     "Adipose CAF": "Fibroblast",
                     "HGF-CAF": "Fibroblast",
                     "TAM1": "Myeloid", 
                     "Myeloid-HighMT": "Unknown/technical", 
                     "Angiogenic EC": "Endothelial", 
                     "Quiescent EC": "Endothelial", 
                     "Venous EC": "Endothelial",
                     "TCD8": "Lymphoid", 
                     "B": "Lymphoid", 
                     "DC": "Myeloid", 
                     "Hepatic EC": "Endothelial", 
                     "Kupffer cells": "Myeloid", 
                     "NK": "Lymphoid", 
                     "Treg": "Lymphoid", 
                     "StrMus-HighMT": "Unknown/technical", 
                     "T-HighMT": "Unknown/technical", 
                     "Mast": "Myeloid", 
                     "Adipocytes": "Stromal/Muscle", 
                     "Endo-HighMT": "Unknown/technical"}

adata.obs = pd.concat([adata.obs,refined_annotations],axis=1)
adata.obs["highlevel_refined"] = adata.obs.refined_annotations.replace(highlevel_refined)

In [ ]:
cNMF_scores_wtop = pd.read_csv("/add/path/here/adata_cNMF_scores_wtop.csv",index_col=0)

In [ ]:
cNMF_scores_wtop = cNMF_scores_wtop.drop(["highlevel_refined","sample_id"],axis=1)

In [ ]:
adata.obs = pd.concat([adata.obs,cNMF_scores_wtop],axis=1)

In [ ]:
import os
sctherapy_res_dir = "/add/path/here/auxiliary_data/scTherapy-results"
os.makedirs(sctherapy_res_dir, exist_ok=True)

# Create malignant-specific DEG

In [ ]:
adata.obs["Malignant_status"] = adata.obs["highlevel_refined"].apply(lambda x: "Malignant" if x=="Carcinoma" else "Healthy")

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="Malignant_status")

In [ ]:
malignant_deg = sc.get.rank_genes_groups_df(adata, group="Malignant").set_index("names")

In [ ]:
malignant_deg = malignant_deg[["logfoldchanges","pvals_adj"]].rename(columns={"logfoldchanges": "avg_log2FC", "pvals_adj": "p_val_adj"})

In [ ]:
malignant_deg.to_csv(os.path.join(sctherapy_res_dir,"scTherapy-malignant-deg.csv"))

# Create state-specific DEG

In [ ]:
adata.obs["Malignant_status_wtop"] = adata.obs["highlevel_wtop"].apply(lambda x: x if x in ["Carcinoma","cNMF_1","cNMF_2","cNMF_3","cNMF_4","cNMF_5"] else "Healthy")

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="Malignant_status_wtop")

In [ ]:
malignant_prog_deg = {}
for ct in [f"cNMF_{i}" for i in range(1,6)]:
    malignant_prog_deg[ct] = sc.get.rank_genes_groups_df(adata, group=ct).set_index("names")

In [ ]:
for ct in malignant_prog_deg:
    malignant_prog_deg[ct] = malignant_prog_deg[ct][["logfoldchanges","pvals_adj"]].rename(columns={"logfoldchanges": "avg_log2FC", "pvals_adj": "p_val_adj"})
    malignant_prog_deg[ct].to_csv(os.path.join(sctherapy_res_dir,f"scTherapy-{ct}-deg.csv"))

# Step in R: predict monotherapy response

In [ ]:
## See R script

# Find differential sensitivity and potential combination therapy

In [ ]:
import os
sctherapy_res_dir = "/add/path/here/auxiliary_data/scTherapy-results"

In [ ]:
sctherapy_results = {}
for ct in np.append(["malignant"], [f"cNMF_{i}" for i in range(1,6)]):
    sctherapy_results[ct] = pd.read_csv(os.path.join(sctherapy_res_dir,f"monotherapy-{ct}-results.csv"),index_col=0)

In [ ]:
high_resp = {}
for ct in sctherapy_results:
    if ct in "malignant":
        continue
    high_resp[ct] = set(sctherapy_results[ct][sctherapy_results[ct]["Response"].isin(["High","High-to-moderate"])].Drug_Name.to_numpy())

In [ ]:
from upsetplot import UpSet
import matplotlib.pyplot as plt

In [ ]:
# Convert the dictionary to a DataFrame format suitable for the UpSet plot
# Create a set of all names across all keys
all_names = set.union(*high_resp.values())

# Create a dictionary that tracks the intersections of the sets
intersections = {}
for name in all_names:
    # For each name, create a tuple where each element is 1 if the name is in the corresponding set, otherwise 0
    intersections[name] = tuple(1 if name in high_resp[key] else 0 for key in high_resp)

# Convert to a DataFrame, where the rows represent names and columns represent the sets
df = pd.DataFrame.from_dict(intersections, orient='index', columns=high_resp.keys())

# Count the occurrences of each unique combination of sets
counts = df.groupby(list(df.columns)).size()

# Create the UpSet plot
upset = UpSet(counts, sort_by='cardinality')
upset.plot()

# Display the plot
plt.title("Intersection of predicted drug sensitivity by state")
plt.savefig("figures/scTherapy-upset.svg", dpi=200, bbox_inches="tight")

In [ ]:
df.sum(axis=1).sort_values()

In [ ]:
combination = []
for i,drug1 in enumerate(df.index):
    for j,drug2 in enumerate(df.index):
        if j>i:
            if ((df.loc[drug1] + df.loc[drug2])>0).sum()==5:
                combination.append(f"{drug1}+{drug2}")

In [ ]:
sctherapy_results["malignant"]

In [ ]:
combination